# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/architgupta18/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This is a provisional Week 1 choice. I can confirm or change the lane after stronger data and validation work through Week 4.

## 1. My lane (or freestyle) and why

**Provisional lane: Refresh / Content Opportunity Scoring.** I will investigate how to rank existing content pages for human review when an editor cannot inspect every page. The starter data is already at the page level and includes observed search visibility, recent activity, content freshness, position, and engagement signals. That makes a ranked review queue more useful than a single global summary: it can help focus limited editorial time on pages with both evidence of opportunity and enough traffic for a review to matter. In later weeks, I will test whether a transparent rule baseline is sufficient before claiming that a more complex model adds value.

In [1]:
from pathlib import Path
import csv

DATA_PATH = Path.cwd() / 'data/raw/content_refresh_anonymized.csv'
if not DATA_PATH.exists():
    DATA_PATH = Path.cwd().parent.parent / 'data/raw/content_refresh_anonymized.csv'
with DATA_PATH.open(newline='') as file:
    df = list(csv.DictReader(file))
print(f"Loaded starter data: {len(df):,} pages across {len({row['client_id'] for row in df}):,} pseudonymized clients.")


Loaded starter data: 30,000 pages across 32 pseudonymized clients.


## 2. The question: decision, action, cost of a wrong call

**Search question.** For a content editor deciding **which existing page to review first this week**, can observed page-level search, freshness, and engagement signals produce an evidence-backed **ranked review queue**? The unit of analysis is one pseudonymized content page in the starter dataset; the eventual output is a page-level priority score with short, human-readable reason codes.

**Decision and action.** An editor or content strategist would take the top pages from the queue, inspect the page and its search context, then decide whether to refresh it, improve its title/snippet, expand or restructure it, or leave it unchanged. The queue supports review; it does not automatically prescribe publishing changes.

**Cost of a wrong call.** A false positive wastes scarce editorial time and can displace a better candidate. A false negative leaves a worthwhile page unreviewed, with possible lost search traffic or engagement. For that reason, I will care about the quality of the highest-ranked pages (for example, precision at a fixed review capacity), clear reason codes, and comparison with a simple rule baseline—not only an overall model score.

**Why data/ML may help.** The signals can interact: a stale page with visible impressions, a weakening recent trend, a workable search position, and low engagement may be more actionable together than any one threshold suggests. A rule may be enough; ML is only worth using if it improves a held-out, decision-relevant ranking beyond that transparent baseline.

In [2]:
# These are descriptive counts from the supplied starter snapshot, not future outcomes.
declining = [row['trend_direction'] == 'down' for row in df]
visible_declining = [
    row['trend_direction'] == 'down' and float(row['impressions_90d']) >= 500
    for row in df
]
low_ctr_visible = [
    float(row['impressions_90d']) >= 500
    and 0 < float(row['avg_position']) <= 20
    and 0 < float(row['ctr']) < 0.5
    for row in df
]

print(f'Current-window decline proxy: {sum(declining):,} / {len(df):,} pages ({sum(declining) / len(df):.1%}).')
print(f'Declining pages with at least 500 impressions in 90 days: {sum(visible_declining):,} ({sum(visible_declining) / len(df):.1%} of all pages).')
print(f'Low-CTR visible review candidates: {sum(low_ctr_visible):,} ({sum(low_ctr_visible) / len(df):.1%} of all pages).')


Current-window decline proxy: 16,262 / 30,000 pages (54.2%).
Declining pages with at least 500 impressions in 90 days: 9,961 (33.2% of all pages).
Low-CTR visible review candidates: 8,543 (28.5% of all pages).


## 3. Quick look at the data (2-3 real numbers)

The executed cells show the starter snapshot contains **30,000 pages from 32 pseudonymized clients**. It also contains **16,262 pages (54.2%)** tagged `down` by the current-window trend rule, including **9,961 pages (33.2% of all pages)** with at least 500 search impressions in the same 90-day window. Finally, **8,543 pages (28.5%)** meet a conservative visible/low-CTR screen (at least 500 impressions, measured position 1–20, and CTR below 0.5%; rate fields are already percentages). These counts are large enough that manual, unranked review would be difficult, and they motivate testing a prioritization method. They do not show that editing any individual page will cause growth.

In [3]:
assert len({row['content_id'] for row in df}) == len(df), 'The starter data should have one row per content page.'
assert all(float(row['impressions_90d']) >= 1 for row in df), 'The supplied starter slice should contain observed impressions.'
print('Sanity checks passed: one row per content_id; rate interpretation retained (CTR < 0.5 means < 0.5%).')


Sanity checks passed: one row per content_id; rate interpretation retained (CTR < 0.5 means < 0.5%).


## 4. Careful words: what I can and can't claim

This work can describe **observed**, page-level patterns in this anonymized starter snapshot and produce **directional, decision-support** rankings for a human review process. The starter `down` field is a proxy derived from current 30-day comparison windows, not an independently observed future outcome; I will not use `trend_direction` or `trend_pct` as model features. A later capstone version should define features before a decision date and measure a future window, with client-holdout or time-aware validation.

I cannot claim that the score proves an edit caused traffic to grow, that it predicts Google, or that every recommended page should be changed. The data contains pseudonymized IDs and aggregated metrics, not client names, URLs, or raw queries. Recommendations must remain review candidates, and any learned approach must earn its place against a simple baseline on held-out data.

In [4]:
forbidden_feature_columns = {'trend_direction', 'trend_pct'}
assert forbidden_feature_columns.issubset(df[0])
print('Leakage guard recorded: trend_direction and trend_pct are excluded from future feature sets.')


Leakage guard recorded: trend_direction and trend_pct are excluded from future feature sets.


## Self-check

- [x] Every required section is filled with markdown framing and reproducible code.
- [x] The notebook was executed top to bottom without errors.
- [x] No client names, URLs, or private queries are included.
- [x] Claims are limited to observed, measured, directional, decision-support results.
- [x] Saved under `work/notebooks/` and ready to commit.